In [39]:
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
import datadict_prep
import importlib
import datetime

In [247]:
importlib.reload(datadict_prep)
df_list = pd.read_excel(Path("Raw DataDict/RE__Requesting_Data_Dictionary_metadata_for_EOCRU_projects_in_CliRes_/"+"IBIS_JKT_V1_DataDictionary.xls"),sheet_name=None)
res = datadict_prep.process_clires_datadict(df_list)
datadict_prep.save_dflist_to_excel(res,"Processed DataCompanion/DataCompanion_IBISRAW_incomplete.xlsx")

df_list = pd.read_excel(Path("Raw DataDict/RE__Requesting_Data_Dictionary_metadata_for_EOCRU_projects_in_CliRes_/"+"EFEK _08IND_V1_DataDictionary.xls"),sheet_name=None)
res = datadict_prep.process_clires_datadict(df_list)
datadict_prep.save_dflist_to_excel(res,"Processed DataCompanion/DataCompanion_EFEKRAW_incomplete.xlsx")

df_list = pd.read_excel(Path("Raw DataDict/RE__Requesting_Data_Dictionary_metadata_for_EOCRU_projects_in_CliRes_/"+"ISARIC_nCoV_P1_DataDictionary.xls"),sheet_name=None)
res = datadict_prep.process_clires_datadict(df_list)
datadict_prep.save_dflist_to_excel(res,"Processed DataCompanion/DataCompanion_ISARICnCoVRAW_incomplete.xlsx")
# df_list.keys()

c:\Users\eocru-lp010\Documents\W\EOCRU\Projects\Data Dictionaries\datadict_prep.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  oriseq_df.loc[:,"FORMNAME"] = "_".join([srs["CRF"],gridinfo["Grid"]])
c:\Users\eocru-lp010\Documents\W\EOCRU\Projects\Data Dictionaries\datadict_prep.py:158: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  oriseq_df.loc[:,"idx"] = tmp[tmp["Grid"] == gridinfo["Grid"]].index.min() - 0.5001
C:\Users\eocru-lp010\AppData\Local\Programs\Python\Python310\lib\site-packages\bs4\__init__

In [136]:
def process_redcap_datadict_longitudinal(df,output,is_long=True):
    rows_with_html = ["Section Header","Field Label","Choices, Calculations, OR Slider Labels"]
    df[rows_with_html] = df[rows_with_html].astype(str).applymap(lambda txt: BeautifulSoup(txt,"lxml").text).replace("nan",np.nan)
    
    #Rename field types =========================================================================================
    df.loc[(df["Field Type"] == "text") & (df["Text Validation Type OR Show Slider Number"].isna()),"Field Type"] = "TEXT"
    df.loc[(df["Field Type"] == "notes"),"Field Type"] = "TEXT"
    df.loc[(df["Field Type"] == "text") & (df["Text Validation Type OR Show Slider Number"] == "integer"),"Field Type"] = "INT"
    df.loc[(df["Field Type"] == "text") & (df["Text Validation Type OR Show Slider Number"] == "number"),"Field Type"] = "DOUBLE"
    df.loc[(df["Field Type"] == "text") & (df["Text Validation Type OR Show Slider Number"] == "date_dmy"),"Field Type"] = "DATE"
    df.loc[(df["Field Type"] == "text") & (df["Text Validation Type OR Show Slider Number"] == "datetime_dmy"),"Field Type"] = "DATETIME"
    df.loc[(df["Field Type"] == "text") & (df["Text Validation Type OR Show Slider Number"] == "datetime_ymd"),"Field Type"] = "DATETIME"
    df.loc[(df["Field Type"] == "text") & (df["Text Validation Type OR Show Slider Number"] == "time"),"Field Type"] = "TIME"
    df.loc[(df["Field Type"] == "text") & (df["Text Validation Type OR Show Slider Number"] == "alpha_only"),"Field Type"] = "LETTERS"
    df.loc[(df["Field Type"] == "radio"),"Field Type"] = "CATEGORIC"
    df.loc[(df["Field Type"] == "dropdown"),"Field Type"] = "CATEGORIC"
    df.loc[(df["Field Type"] == "yesno"),"Choices, Calculations, OR Slider Labels"] = "1, Yes | 0, No"
    df.loc[(df["Field Type"] == "yesno"),"Field Type"] = "CATEGORIC"
    df.loc[(df["Field Type"] == "slider"),"Text Validation Type OR Show Slider Number"] = np.nan
    df.loc[(df["Field Type"] == "slider"),"Choices, Calculations, OR Slider Labels"] = np.nan
    df.loc[(df["Field Type"] == "slider"),"Field Type"] = "DOUBLE"
    df.loc[(df["Field Type"] == "calc"),"Choices, Calculations, OR Slider Labels"] = np.nan
    df.loc[(df["Field Type"] == "calc"),"Field Type"] = "DOUBLE"

    #Process Checkboxes =========================================================================================
    df_cbox = df[(df["Field Type"] == "checkbox")]
    for idx, row in df_cbox.iterrows():
        cbox_options = strchoice_to_list(row["Choices, Calculations, OR Slider Labels"])
        dict_orig = {k:[v] * len(cbox_options) for k,v in row.to_dict().items()}
        dict_cbox = {"Variable / Field Name":["___".join([row["Variable / Field Name"],l[0].lower()]) for l in cbox_options],
                     "Field Type":["CATEGORIC"] * len(cbox_options),
                     "Field Label":[" ".join([row["Field Label"]," (",l[1],")"]) for l in cbox_options],
                     "Choices, Calculations, OR Slider Labels" : ["1, Yes | 0, No"] * len(cbox_options),
                     "Matrix Group Name" : [row["Variable / Field Name"]] * len(cbox_options)}
        dict_orig.update(dict_cbox)
        df_cbox_long = pd.DataFrame(data=dict_orig,columns=df.columns,index=[x/1000 for x in range(idx*1000+1,idx*1000+len(cbox_options)+1)])
        # print(df_cbox_long)
        df = pd.concat([df,df_cbox_long])
        df.drop(index=idx,inplace=True)
    df.sort_index(inplace=True)

    #Add UNUSED column =========================================================================================
    # df["UNUSED"] = (df["Field Type"] == "descriptive") & (df["Identifier?"] == "y")
    df = df.assign(UNUSED       = lambda x: ((x["Field Type"] == "descriptive") | (x["Identifier?"] == "y") |  \
                                            (x["Field Annotation"].astype(str).str.find("@CALCTEXT") >= 0) | \
                                            (x["Field Annotation"].astype(str).str.find("@HIDDEN") >= 0) | \
                                            (x["Field Annotation"].astype(str).str.find("eocru:data:hidden") >= 0) | \
                                            (x["Variable / Field Name"].astype(str).str.find("_calc") >= 0) | \
                                            (x["Variable / Field Name"].astype(str).str.find("_text") >= 0)) &
                                            ~(x["Field Annotation"].astype(str).str.find("eocru:data:accept") >= 0),
                   ISINSTANCE = False)
                #    HAS_HIDDEN   = lambda x: (x["Field Annotation"].astype(str).str.find("@HIDDEN") >= 0))
    df[["Identifier?","Required Field?"]] = df[["Identifier?","Required Field?"]].replace({"y":True})

    #Fix header =========================================================================================
    if is_long:
        dict_longitudinal_start = {"Variable / Field Name":["redcap_event_name","redcap_repeat_instrument","redcap_repeat_instance","redcap_data_access_group"],
                                "Field Type":["META","META","METAINT","META"],
                                "Field Label":["Event Name","Repeat Form Name","Repeat Form No.","Site Name"],
                                "UNUSED":[False]*4,
                                "ISINSTANCE":[False,False,True,False]}
        df_longitudinal_start = pd.DataFrame(data=dict_longitudinal_start,columns=df.columns,index=[0.1,0.2,0.3,0.4])
        df = pd.concat([df,df_longitudinal_start]).sort_index()
    df.loc[0,"Field Type"] = "KEY"

    
    #Change CategoricCodes to JSON format  =========================================================================================
    df.loc[(df["Field Type"] == "CATEGORIC") & (df["Choices, Calculations, OR Slider Labels"].notna()),"Choices, Calculations, OR Slider Labels"] = \
        df.loc[(df["Field Type"] == "CATEGORIC") & (df["Choices, Calculations, OR Slider Labels"].notna()),"Choices, Calculations, OR Slider Labels"].astype(str).apply(strchoice_to_json)

    df = (df.rename(columns={"Variable / Field Name":"VARNAME",
            "Form Name":"FORMNAME",
            "Field Type":"DATATYPE",
            "Field Label":"DEFINITION",
            "Choices, Calculations, OR Slider Labels":"CATEGORICCODES",
            "Identifier?":"IDENTIFIER",
            "Branching Logic (Show field only if...)":"BRANCHLOGIC",
            "Required Field?":"REQUIRED",
            "Field Annotation":"NOTES",
            "Matrix Group Name":"ANALYSISGROUP"})
            .drop(columns=["Section Header","Field Note","Text Validation Min","Text Validation Max", \
                "Question Number (surveys only)","Custom Alignment","Text Validation Type OR Show Slider Number", \
                "Matrix Ranking?"]))
    df.to_csv(output,index=False, encoding="utf-8-sig")
    return

def strchoice_to_json(text):
    lis = text.split("|")
    lis = [l.split(",",1) for l in lis]
    # print(lis)
    lis = {l[0].strip() : l[1].strip() for l in lis}
    return json.dumps(lis)
def strchoice_to_list(text):
    lis = text.split("|")
    lis = [l.split(",",1) for l in lis]
    lis = [[a.strip() for a in l] for l in lis]
    return lis

In [6]:
importlib.reload(datadict_prep)
#SSAT
df = pd.read_csv(Path.cwd().parent / "Data Dictionaries" / "Raw DataDict" / "SSAT" / "SSAT_DataDictionary_2022-06-09.csv")
df_events = pd.read_csv(Path.cwd().parent / "Data Dictionaries" / "Raw DataDict" / "SSAT" / "SSAT_Events_2022-03-30.csv")
df_crfevent = pd.read_csv(Path.cwd().parent / "Data Dictionaries" / "Raw DataDict" / "SSAT" / "SSAT_InstrumentDesignations_2022-03-30.csv")
savename = Path.cwd().parent / "Data Dictionaries" / ("Processed DataCompanion/DataCompanion_SSATRAW_incomplete_"+ datetime.datetime.now().strftime("%Y%m%d_%H%M%S")+".xlsx"
res2 = datadict_prep.process_redcap_datadict_longitudinal(df,df_events,df_crfevent)
datadict_prep.save_dflist_to_excel(res2,savename))

#MetLep
df = pd.read_csv(Path.cwd().parent / "Data Dictionaries" / "Raw DataDict" / "MetLep" /"MetLepTrial_DataDictionary_2022-03-19.csv")
df_events = pd.read_csv(Path.cwd().parent / "Data Dictionaries" / "Raw DataDict" / "MetLep" /"MetLepTrial_Events_2022-03-30.csv")
df_crfevent = pd.read_csv(Path.cwd().parent / "Data Dictionaries" / "Raw DataDict" / "MetLep" /"MetLepTrial_InstrumentDesignations_2022-03-21.csv")
savename = Path.cwd().parent / "Data Dictionaries" / ("Processed DataCompanion/DataCompanion_MetLepRAW_incomplete_"+ datetime.datetime.now().strftime("%Y%m%d_%H%M%S")+".xlsx"
res2 = datadict_prep.process_redcap_datadict_longitudinal(df,df_events,df_crfevent)
datadict_prep.save_dflist_to_excel(res2,savename)

# #INVITE OUCRU
# df = datadict_prep.fetch_datadict_api("INVITE OUCRU")
# df_events = pd.read_csv(Path("Raw DataDict/INVITE 2 OUCRU/"+"INVITE_Events_2022-03-31.csv"))
# df_crfevent = pd.read_csv(Path("Raw DataDict/INVITE 2 OUCRU/"+"INVITE_InstrumentDesignations_2022-03-31.csv"))
# res2 = datadict_prep.process_redcap_datadict_longitudinal(df,df_events,df_crfevent)
# datadict_prep.save_dflist_to_excel(res2,"Processed DataCompanion/DataCompanion_INVITE2RAW_incomplete.xlsx")



In [93]:
importlib.reload(datadict_prep)
importlib.reload(redcap_apimodules)
redcapdata = datadict_prep.fetch_redcap_all_projectinfo("INVITE OUCRU")

In [192]:
pd.DataFrame(redcapdata['dag'])

,data_access_group_name,unique_group_name
0,(01) RSUD PASAR MINGGU,01_rsud_pasar_ming
1,(02) RS ST. CAROLOUS,02_rs_st_carolous
2,(03) PUSKESMAS CAKUNG,03_puskesmas_cakun
3,(04) PUSKESMAS CIRACAS,04_puskesmas_cirac
4,(05) PUSKESMAS DUREN SAWIT,05_puskesmas_duren


In [153]:
importlib.reload(datadict_prep)
importlib.reload(redcap_apimodules)
redcapdata, metadf = datadict_prep.fetch_raw_data("INVITE OUCRU")

In [154]:
redcapdata['data'].columns

Index(['record', 'redcap_event_name', 'redcap_repeat_instrument',
       'redcap_repeat_instance', 'field_name', 'value'],
      dtype='object')

In [193]:

# redcapdata['data']['redcap_event_name'].unique()

,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,field_name,value
record,,,,,
112-22,Extra Visit(s),Extra Visit,1,exvis_serum,Yes


In [176]:
def get_varcol_from_eav(eavfile,fname,eventname = None,repeatinstrument = None, repeatinstance = None):
    conditions = {}
    # print(eavfile['record'].unique())
    cond_true = pd.Series({a:True for a in eavfile['record'].unique()})
    conditions[0] = (eavfile.set_index("record").field_name == fname)
    if eventname != None:
        conditions[1] = (eavfile.set_index("record").redcap_event_name == eventname)
    else:
        conditions[1] = cond_true
    if repeatinstrument != None:
        conditions[2] = (eavfile.set_index("record").redcap_repeat_instrument == repeatinstrument)
    else:
        conditions[2] = cond_true
    if repeatinstance != None:
        conditions[3] = (eavfile.set_index("record").redcap_repeat_instance == repeatinstance)
    else:
        conditions[3] = cond_true
    return eavfile.set_index("record")[conditions[0] & conditions[1] & conditions[2] & conditions[3]]["value"]

In [189]:
df2_eav = redcapdata['data']
# df2_eav = rcdf2['data']
df2_patlist = pd.DataFrame(index=df2_eav['record'].unique().tolist())
df2_patlist['h_dag'] = get_varcol_from_eav(df2_eav,"pmi_dag_default")
df2_patlist['h_cohort'] = get_varcol_from_eav(df2_eav,"eli_vaccpath")
df2_patlist['h_V1'] = get_varcol_from_eav(df2_eav,"vac_manufacturer","Vaccine #1")
df2_patlist['h_V2'] = get_varcol_from_eav(df2_eav,"vac_manufacturer","Vaccine #2")
df2_patlist['h_V3'] = get_varcol_from_eav(df2_eav,"vac_manufacturer","Vaccine #3")
# pd.to_datetime(get_varcol_from_eav(df2_eav,"eli_date_enrol"))
df2_patlist
get_varcol_from_eav(df2_eav,"exvis_serum")

record
112-22    Yes
Name: value, dtype: object

In [216]:
n_days = 30
(tmp_date >= datetime.datetime.now() - pd.Timedelta(n_days,unit="d")).sum()

50

In [11]:
importlib.reload(datadict_prep)
#INVITE OUCRU
df = pd.read_csv(Path("Raw DataDict/INVITE OX/"+"INVITE_OX_DataDictionary_2022-03-31.csv"),encoding="iso-8859-1")
res2 = datadict_prep.process_redcap_datadict_short(df)
datadict_prep.save_dflist_to_excel(res2,"Processed DataCompanion/DataCompanion_INVITE1RAW_incomplete.xlsx")

In [22]:
df = pd.read_csv("INVITE_OX_DataDictionary_2022-03-09.csv",encoding="iso-8859-1")
datadict_prep.process_redcap_datadict_longitudinal(df,False)

# cleantext = BeautifulSoup(raw_txt,"lxml").text